# Specifications

In [1]:
import SLiCAP as sl
from sympy import pi, sqrt, log, Abs
sl.initProject('Hearing loop', notebook=True)

Compiling library: SLiCAP.lib.


Compiling library: SLiCAPmodels.lib.


## Functional model
Below the functional model of the hearing loop receive coil amplifier with source, load and noise weighting function.

In [2]:
cir = sl.makeCircuit("kicad/A1/fumo.kicad_sch")
sl.img2html("fumo.svg", 800)

Creating netlist of kicad/A1/fumo.kicad_sch using KiCAD


Creating drawing-size SVG and PDF images of kicad/A1/fumo.kicad_sch


Plotted to '/home/danieltyukov/workspace/tud/tud-structured-electronic-design/Notebooks/img/fumo.svg'.
Done.


Checking netlist: cir/fumo.cir


## System and component specifications
The following specifications are derived from the application diagram and the component specifications.

In [3]:
# Performance specifications
SPL             = 20E-6     # 0dB SPL level in Pa
n_SPL           = 30        # Allowed SPL noise level
SPL_max         = 110       # Peak sound pressure level 
f_min           = 600       # Low -3dB cut-off frequency
f_max           = 6000      # High -3dB cut-off frequency
f_fp            = 5000      # Maximum full-power frequency

# Interface specifications
V_ADC_high      = 0.9      # Full-scale input voltage ADC
V_ADC_low       = 0        # Zero-scale input voltage ADC
A_mic           = -35.5    # Typical microphone sensitivity in dBV/Pa
H_ref           = 90       # dB/(A/m) inductive loop reference
f_ref           = 1000     # Reference frequency for receive coil sensitivity
f_res           = 0.15e6   # Minimum receive-coil self-resonant frequency
A_L             = -59.4    # dBV/(A/m) typical open-circuit receive coil sensitivity at f_ref
Rs              = 875      # Typical receive coil resistance
Ls              = 0.12     # Typical receive coil inductance
CL              = 10e-12   # Load capacitance

# Cost factors
P_max           = 1e-3     # Maximum power dissipation at maximum supply voltage 
V_DD_max        = 1.3      # Maximum supply voltage
V_DD_min        = 1.1      # Minimum supply voltage    

# Environmental conditions
T_min           = 0        # Minimum operating temperature
T_max           = 50       # Maximum operating temperature

specs = []

## Derived specifications

The parasitic parallel receive coil capacitance can be calculated from the self-resonance frequency as:
\begin{equation}
C_s = \frac{1}{\left( 2\pi f_\mathrm{res}\right)^2 L_s}
\end{equation}

In [4]:
Cs              = 1/((2 *pi * f_res)**2 * Ls)
sl.eqn2html("C_s", Cs, units="F")

Damping of this resonance can be achieved by placing a resistor in parallel with the inductor.
For critical damping ($Q=\frac{1}{2}\sqrt{2}$) the resistance $R_t$ can be obtained as
\begin{equation}
R_t=\frac{\sqrt{2}}{4\pi f_\mathrm{res}C_s}
\end{equation}

In [5]:
Rt              = sqrt(2)/(4 * pi * f_res * Cs)
sl.eqn2html("R_t", Rt, units="Ohm")

The DIN-A weighted output noise of the receive coil amplifier may correspond to 30 dBSPL (microphone output):
\begin{equation}
V_\mathrm{onoise} = 20\cdot 10^{-6} 10^{\left( A_\mathrm{mic} + n_\mathrm{SPL}\right)/20}
\end{equation}

In [6]:
V_onoise        = SPL * 10**((A_mic + n_SPL)/20)
sl.eqn2html("V_onoise", V_onoise, units="V")

The peak microphone output voltage at peak SPL is found as

\begin{equation}
V_\mathrm{mic} = 20 \cdot 10^{-6} \cdot 10^{\left( A_\mathrm{mic}+ \mathrm{SPL_{max}}\right)/20}
\end{equation}

In [7]:
V_mic           = SPL * 10**((A_mic + SPL_max)/20)
sl.eqn2html("V_mic", V_mic, units="V")

The peak receive coil output voltage at reference frequency is
\begin{equation}
V_i = 10^{\left( A_L + \mathrm{SPL_{max}} - H_\mathrm{ref}\right)/20}
\end{equation}

In [8]:
Vi             = 10**((A_L + SPL_max - H_ref)/20)
sl.eqn2html("V_i", Vi, units="V")

The required integrator time constant, such that the receive-coil amplifier output equals the microphone output, can be calculated as
\begin{equation}
\tau_i = \frac{V_i}{2\pi f_\mathrm{ref} V_\mathrm{mic}}
\end{equation}

In [9]:
tau_i           = Vi/(2*pi*f_ref*V_mic)
sl.eqn2html("tau_i", tau_i, units="s")

The peak-peak receive coil output voltage at maximum full-power frequency is found as
\begin{equation}
V_{i_\mathrm{pp}} = 2 V_i \frac{f_\mathrm{fp}}{f_\mathrm{ref}}
\end{equation}

In [10]:
Vi_pp           = 2 * Vi * f_fp/f_ref
sl.eqn2html("V_i_pp", Vi_pp, units="V")

## Create a SLiCAP design database
We will store the specifications in a "CSV" file. SLiCAP offers **specItem()** objects and can convert a list of these objects into a "CSV" file. During subsequent design steps, we will read specifications from this "CSV" file and add more specifications to it.

The DIN-A noise weighting function is defined by the built-in function **DIN_A()**.

In [11]:
# source impedance specification

specs.append(
    sl.specItem(
        "L_s",
        description = "Source inductance",
        value       = Ls,
        units       = "H",
        specType    = "Interface",
    )
)
specs.append(
    sl.specItem(
        "R_s",
        description = "Source resistance",
        value       = Rs,
        units       = "Omega",
        specType    = "Interface",
    )
)
specs.append(
    sl.specItem(
        "C_s",
        description = "Source capacitance",
        value       = Cs,
        units       = "F",
        specType    = "Interface",
    )
)
specs.append(
    sl.specItem(
        "R_t",
        description = "Source termination resistance for critical damping",
        value       = Rt,
        units       = "Omega",
        specType    = "Interface",
    )
)

# source signal specification

specs.append(
    sl.specItem(
        "Vi_pp",
        description = "Maximum peak-peak input voltage",
        value       = Vi_pp,
        units       = "V",
        specType    = "Interface",
    )
)
specs.append(
    sl.specItem(
        "f_fp",
        description = "Maximum full-power frequency",
        value       = f_fp,
        units       = "Hz",
        specType    = "Interface",
    )
)

# Load impedance specification

specs.append(
    sl.specItem(
        "C_L",
        description = "Load capacitance",
        value       = CL,
        units       = "F",
        specType    = "Interface",
    )
)

# Load signal specification

specs.append(
    sl.specItem(
        "V_ADC_high",
        description = "Full-scale input voltage ADC",
        value       = V_ADC_high,
        units       = "V",
        specType    = "Interface",
    )
)

specs.append(
    sl.specItem(
        "V_ADC_low",
        description = "Zero-scale input voltage ADC",
        value       = V_ADC_low,
        units       = "V",
        specType    = "Interface",
    )
)

# Supply voltage specification
specs.append(
    sl.specItem(
        "V_DD_max",
        description = "Maximum supply voltage",
        value       = V_DD_max,
        units       = "V",
        specType    = "Interface",
    )
)
specs.append(
    sl.specItem(
        "V_DD_min",
        description = "Minimum supply voltage",
        value       = V_DD_min,
        units       = "V",
        specType    = "Interface",
    )
)

# Performance requirements (Transfer, Bandwidth, Resolution)

specs.append(
    sl.specItem(
        "tau_i",
        description = "Integration time constant",
        value       = tau_i,
        units       = "s",
        specType    = "performance",
    )
)
specs.append(
    sl.specItem(
        "f_min",
        description = "-3dB high-pass cut-off frequency",
        value       = f_min,
        units       = "Hz",
        specType    = "performance",
    )
)
specs.append(
    sl.specItem(
        "f_max",
        description = "-3dB low-pass cut-off frequency",
        value       = f_max,
        units       = "Hz",
        specType    = "performance",
    )
)
specs.append(
    sl.specItem(
        "V_onoise",
        description = "DIN A weighted output voltage noise",
        value       = V_onoise,
        units       = "V",
        specType    = "performance",
    )
)
specs.append(
    sl.specItem(
        "DIN_A",
        description = "Noise weighting function",
        value       = sl.DIN_A(),
        units       = "",
        specType    = "performance",
    )
)
specs.append(
    sl.specItem(
        "f_ref",
        description = "Reference frequency",
        value       = f_ref,
        units       = "Hz",
        specType    = "test",
    )
)

# Costs specification

specs.append(
    sl.specItem(
        "P_max",
        description = "Maximum power dissipation at maximum supply voltage",
        value       = P_max,
        units       = "W",
        specType    = "Costs",
    )
)

# Environmental operating conditions

specs.append(
    sl.specItem(
        "T_min",
        description = "Minimum operating temperature",
        value       = T_min,
        units       = "Celsius",
        specType    = "Environment",
    )
)
specs.append(
    sl.specItem(
        "T_max",
        description = "Maximum operating temperature",
        value       = T_max,
        units       = "Celsius",
        specType    = "Environment",
    )
)

For easy access, you can store the specifications in a dictionary, the keys will be the name of the specItem and the value the specItem object.

In [12]:
spec_dict = sl.specs2dict(specs)
for key in spec_dict.keys():
    print(key, spec_dict[key].value)

L_s 3/25
R_s 875
C_s 462962962962963/(5000000000000000000000000*pi**2)
R_t 18000*sqrt(2)*pi
Vi_pp 2143038610475213/20000000000000000
f_fp 5000
C_L 1/100000000000
V_ADC_high 9/10
V_ADC_low 0
V_DD_max 13/10
V_DD_min 11/10
tau_i 50459159092039/(1000000000000000000*pi)
f_min 600
f_max 6000
V_onoise 10617688884619767/1000000000000000000000
DIN_A 18719114681919*f**4/(100000*sqrt((f**2 + 1159929/100)*(f**2 + 54449641/100))*(f**2 + 10609/25)*(f**2 + 148693636))
f_ref 1000
P_max 1/1000
T_min 0
T_max 50


Display a nicely rendered output of all specifications using **specs2html(**\<list with specItems\>**)**:

In [13]:
sl.specs2html(specs)

## Verification


Specification items that carry the same names as circuit parameters can be assigned to their corresponding circuit parameter using **specs2circuit(**\<list of specItems\>, \<circuit object\>**)**.

After we assigned the parameter values to the circuit elements, we check if all circuit elements have numeric values using **elementData2html(**\<circuit object\>**)**. If so, we can plot the transfer of the intended circuit versus frequency.

In [14]:
sl.specs2circuit(specs, cir)
sl.elementData2html(cir)

RefDes,Nodes,Refs,Model,Param,Symbolic,Numeric
C1,1 0,,C,value,$C_{s}$,$9.382\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C2,out 0,,C,value,$C_{L}$,$10\cdot 10^{-12}$
,,,,vinit,$0$,$0$
E1,out 0 1 0,,E,value,$\frac{1}{s \tau_{i}}$,$\frac{6.226 \cdot 10^{4}}{s}$
E2,noise 0 out 0,,E,value,$DIN_{A}$,$\frac{1.872 \cdot 10^{8} f^{4}}{\left(\left(f^{2} + 1.16 \cdot 10^{4}\right) \left(f^{2} + 5.445 \cdot 10^{5}\right)\right)^{0.5} \left(f^{2} + 424.4\right) \left(f^{2} + 1.487 \cdot 10^{8}\right)}$
L1,1 2,,L,value,$L_{s}$,$0.12$
,,,,iinit,$0$,$0$
R1,2 3,,R,value,$R_{s}$,$875$
,,,,noisetemp,$T$,$300$


### Frequency characteristic
The magnitude characteristic shows the signal integration and the critical damping at resonance.

In [15]:
transfer = sl.doLaplace(cir, pardefs="circuit", numeric=True)
sl.plotSweep("fumo_dBmag", "Magnitude of the transfer", transfer, 0.1, 1e3, 200, sweepScale="k", funcType="dBmag")
sl.img2html("fumo_dBmag.svg", 600)

In [16]:
sl.plotSweep("fumo_phase", "Phase of the transfer", transfer, 0.1, 1e3, 200, sweepScale="k", funcType="phase")
sl.img2html("fumo_phase.svg", 600)

The gain at the reference frequency must be:
\begin{equation}
A_\mathrm{A1} = 20\log{\frac{V_\mathrm{mic}}{V_i}}
\end{equation}

In [17]:
sl.eqn2html("f_ref", f_ref, units="Hz")

In [18]:
sl.eqn2html("A_A1", 20*log(V_mic/Vi)/log(10), units="dB")

We will check if this matches the actual value.

The actual value is obtaines from the simulation result:

In [19]:
gain = 20*log(Abs(transfer.laplace.subs(sl.ini.laplace, 2j*pi*f_ref)))/log(10)
sl.eqn2html("A_fumo", gain, units = "dB")

**Questions**

1. What's the cause of the difference?
2. Please check your answer!

### Noise budget taken by signal source
We will now evaluate the DIN_A weighted RMS output noise generated by the receive coil and its termination. If it exceeds the budget, there's no need to continue ...

In [20]:
RMSnoise = sl.rmsNoise(sl.doNoise(cir, pardefs="circuit", numeric=True, detector="V_noise"), "onoise", f_min, f_max)
sl.eqn2html("Vn_fumo", RMSnoise, units= "V")

The relative part of the output noise taken by the source is:

In [21]:
sl.expr2html(RMSnoise/V_onoise)

In [22]:
specs.append(
    sl.specItem(
        "V_ns",
        description = "Weighted noise output noise contribution of Rs",
        value       = RMSnoise,
        units       = "V",
        specType    = "Interface",
    )
)
sl.specs2csv(specs, 'A1specs.csv')

**Task:**
1. Draw your own conclusions!